# TradingAgents Notebook Analysis

This notebook demonstrates how to run the TradingAgents analysis directly from Python without using the CLI.

## Setup

First, make sure you have the required dependencies installed and your environment variables configured.

In [ ]:
# Import required modules
from cli.notebook_interface import (
    NotebookAnalysisConfig,
    AnalystSelection,
    ResearchDepth,
    run_notebook_analysis,
    display_notebook_report,
)
from pathlib import Path
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

print("✓ Imports successful!")

## Step 1: Configure Your Analysis

Define your analysis parameters programmatically. You can customize:
- **Ticker**: Stock symbol to analyze
- **Analysis Date**: When to analyze (YYYY-MM-DD format)
- **Analysts**: Which analyst teams to use
- **Research Depth**: How thorough the analysis should be
- **LLM Provider & Models**: Which AI model to use

In [ ]:
# Example 1: Analyze Apple with all analysts at medium depth
config = NotebookAnalysisConfig(
    ticker="AAPL",
    analysis_date="2024-01-15",
    selected_analysts=[
        AnalystSelection.MARKET,
        AnalystSelection.SOCIAL,
        AnalystSelection.NEWS,
        AnalystSelection.FUNDAMENTALS,
    ],
    research_depth=ResearchDepth.MEDIUM,
    llm_provider="openai",
    backend_url="https://api.openai.com/v1",
    shallow_thinker="gpt-4.1",
    deep_thinker="gpt-5",
    openai_reasoning_effort="high",
)

print(f"✓ Configuration created for {config.ticker}")
print(f"  Analysts: {', '.join(a.value for a in config.selected_analysts)}")
print(f"  Research Depth: {config.research_depth.name}")
print(f"  Date: {config.analysis_date}")

## Step 2: Run the Analysis

Execute the analysis with your configuration. The function will stream results and display progress.

In [ ]:
# Run the analysis
final_state, report_text = run_notebook_analysis(
    config=config,
    show_progress=True,
    save_report=True,
    save_path=Path("./reports") / config.ticker / config.analysis_date,
)

print("\n✓ Analysis complete!")
print(f"  Report sections available: {list(final_state.keys())}")

## Step 3: View the Results

Display the complete analysis report with formatted sections.

In [ ]:
# Display the full report
display_notebook_report(final_state)

## Step 4: Access Individual Reports (Optional)

You can access specific sections of the analysis programmatically.

In [ ]:
# Access individual reports
if final_state.get("market_report"):
    print("=== Market Analysis ===")
    print(final_state["market_report"])
    print()

if final_state.get("sentiment_report"):
    print("=== Social Sentiment Analysis ===")
    print(final_state["sentiment_report"])
    print()

if final_state.get("news_report"):
    print("=== News Analysis ===")
    print(final_state["news_report"])
    print()

if final_state.get("fundamentals_report"):
    print("=== Fundamentals Analysis ===")
    print(final_state["fundamentals_report"])

## Example 2: Batch Analysis

Run analysis on multiple tickers with the same configuration.

In [ ]:
# Example: Analyze multiple tickers
tickers = ["AAPL", "MSFT", "GOOGL"]
results = {}

for ticker in tickers:
    print(f"\nAnalyzing {ticker}...")
    
    batch_config = NotebookAnalysisConfig(
        ticker=ticker,
        selected_analysts=[AnalystSelection.MARKET, AnalystSelection.NEWS],
        research_depth=ResearchDepth.SHALLOW,  # Use shallow for batch
        llm_provider="openai",
        shallow_thinker="gpt-4.1",
        deep_thinker="gpt-5",
    )
    
    final_state, report_text = run_notebook_analysis(
        config=batch_config,
        show_progress=False,
    )
    
    results[ticker] = final_state
    print(f"✓ {ticker} analysis complete")

print(f"\n✓ Batch analysis complete! Analyzed {len(results)} tickers")

## Example 3: Custom Configuration

Use different LLM providers and models.

In [ ]:
# Example: Using Google Gemini
gemini_config = NotebookAnalysisConfig(
    ticker="TSLA",
    selected_analysts=[AnalystSelection.MARKET, AnalystSelection.FUNDAMENTALS],
    research_depth=ResearchDepth.DEEP,
    llm_provider="google",
    backend_url="https://generativelanguage.googleapis.com/v1",
    shallow_thinker="gemini-2.5-flash",
    deep_thinker="gemini-3-pro-preview",
    google_thinking_level="high",
)

print("✓ Google Gemini configuration created")
print(f"  Provider: {gemini_config.llm_provider}")
print(f"  Shallow: {gemini_config.shallow_thinker}")
print(f"  Deep: {gemini_config.deep_thinker}")
print(f"  Thinking: {gemini_config.google_thinking_level}")

# Uncomment to run
# final_state, report = run_notebook_analysis(gemini_config)

## Documentation

### NotebookAnalysisConfig
Configure the analysis with these parameters:

- **ticker** (str): Stock ticker (e.g., 'AAPL', 'SPY')
- **analysis_date** (str): Date in YYYY-MM-DD format. Defaults to today.
- **selected_analysts** (List[AnalystSelection]): Which analysts to run
  - `AnalystSelection.MARKET` - Market trends and technical analysis
  - `AnalystSelection.SOCIAL` - Social media sentiment
  - `AnalystSelection.NEWS` - News sentiment and impact
  - `AnalystSelection.FUNDAMENTALS` - Company fundamentals

- **research_depth** (ResearchDepth): Analysis depth
  - `ResearchDepth.SHALLOW` (1) - Quick research
  - `ResearchDepth.MEDIUM` (3) - Balanced (default)
  - `ResearchDepth.DEEP` (5) - Comprehensive

- **llm_provider** (str): LLM service (openai, google, anthropic, xai, openrouter, ollama)
- **backend_url** (str): API endpoint URL
- **shallow_thinker** (str): Model for quick thinking
- **deep_thinker** (str): Model for deep reasoning
- **google_thinking_level** (str): For Gemini (high/minimal)
- **openai_reasoning_effort** (str): For OpenAI (high/medium/low)

### run_notebook_analysis

Execute analysis and return results:

```python
final_state, report_text = run_notebook_analysis(
    config=config,
    show_progress=True,
    save_report=False,
    save_path=None
)
```

Returns:
- **final_state** (Dict): Complete analysis state with all reports
- **report_text** (str): Formatted report as markdown text

### display_notebook_report

Display formatted report in the notebook:

```python
display_notebook_report(final_state)
```